In [1]:
from __future__ import annotations
import argparse
import json
import os
import random
import sys
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Tuple
from loguru import logger
from tqdm.notebook import tqdm

import numpy as np
import torch
from torch.utils.data import DataLoader
from torch_geometric.data import Batch
import matplotlib.pyplot as plt
from data.common import CONS_PAD, VARS_PAD  # noqa: E402
from data.datasets import GraphDataset, PadFeaturesTransform  # noqa: E402
from models.gnn import GNNPolicy  # noqa: E402
from sklearn.decomposition import PCA  # noqa: F401
from sklearn.manifold import TSNE  # noqa: F401
import configargparse
from jobs.utils import set_all_seeds, device_from_args, build_dataloaders
from data.common import ProblemClass

In [2]:
out_dir = Path("./results/pretrained_embeddings")
out_dir.mkdir(parents=True, exist_ok=True)
parser = configargparse.ArgumentParser(
    allow_abbrev=False,
    default_config_files=["../configs/finetune/finetune_ALL_200/finetune_ALL_200_gnn_full_finetune.yml"],
)
parser.add_argument(
        "--encoder_path",
        type=str,
        help="Path to encoder-only checkpoint from pretraining (e.g., .../best_encoder.pt).",
    )
parser.add_argument("--devices", type=str, default="0")
parser.add_argument("--batch_size", type=int, default=8)
parser.add_argument("--num_workers", type=int, default=0)
parser.add_argument("--seed", type=int, default=42)
parser.add_argument("--max_graphs_per_class", type=int, default=7500)
parser.add_argument(
        "--pooling",
        type=str,
        default="mean_all",
        choices=["concat", "mean_all"],
        help="Graph embedding pooling strategy.",
    )
parser.add_argument(
        "--method",
        type=str,
        default="pca_tsne",
        choices=["pca", "tsne", "pca_tsne"],
        help="Dimensionality reduction method to run.",
    )
parser.add_argument("--pca_dim", type=int, default=50, help="PCA dims before t-SNE.")
parser.add_argument("--perplexity", type=float, default=30.0, help="t-SNE perplexity.")
parser.add_argument(
        "--finetune_mode",
        type=str,
        choices=["full", "heads"],
        default="full",
        help="'full' updates encoder+heads; 'heads' freezes encoder and trains heads only.",
    )

d = parser.add_argument_group("data")

d.add_argument(
    "--use_bipartite_graphs", action="store_true", help="Must be set for GNNPolicy."
)
d.add_argument(
    "--problems",
    type=str,
    nargs="+",
    default=[ProblemClass.INDEPENDANT_SET, ProblemClass.COMBINATORIAL_AUCTION, ProblemClass.CAPACITATED_FACILITY_LOCATION, ProblemClass.SET_COVER],
    help="Problem type(s).",
)
d.add_argument(
    "--is_sizes", type=int, nargs="+", default=[200]
)
d.add_argument(
    "--ca_sizes", type=int, nargs="+", default=[100]
)
d.add_argument(
    "--sc_sizes", type=int, nargs="+", default=[200]
)
d.add_argument(
    "--cfl_sizes", type=int, nargs="+", default=[200]
)
d.add_argument(
    "--rnd_sizes", type=int, nargs="+", default=[200]
)

d.add_argument("--n_instances", type=int, default=35000)
d.add_argument("--data_root", type=str, default="../data/instances")
d.add_argument(
    "--val_split",
    type=float,
    default=0.15,
    help="Validation split ratio (default: 0.15 for 70/15/15 train/val/test)",
)
args, _ = parser.parse_known_args()
GNNPolicy.add_args(parser) 
args, _ = parser.parse_known_args()

In [3]:
print("Arguments:", args)

Arguments: Namespace(encoder_path='/home/joachim-verschelde/Repos/KKT_MPNN/src/experiments/kkt_gnn_node_pretraining_experiments/run_gnn_policy-gv=2-lv=8-slice=512-point=13-lmbd=0.256-std=0.0-dim=128-l_mask=0.6-g_mask=0.05-l_e_mask=0.05-g_e_mask=0.6-dp=0.05-l_dim=128-conv=gatv2-heads=4_20260115_175620/best_encoder.pt', devices='0', batch_size=80, num_workers=0, seed=42, max_graphs_per_class=7500, pooling='mean_all', method='pca_tsne', pca_dim=50, perplexity=30.0, finetune_mode='full', use_bipartite_graphs=True, problems=['CA', 'SC', 'IS', 'CFL'], is_sizes=[200], ca_sizes=[100], sc_sizes=[200], cfl_sizes=[200], rnd_sizes=[200], n_instances=50000, data_root='../data/instances', val_split=0.15, lejepa_n_global_views=2, lejepa_n_local_views=8, sigreg_slices=512, sigreg_points=13, lejepa_lambda=0.256, lejepa_std_loss_weight=0.0, embedding_size=128, cons_nfeats=4, edge_nfeats=1, var_nfeats=18, num_emb_type='periodic', num_emb_bins=32, num_emb_freqs=16, lejepa_local_mask=0.6, lejepa_global_mas

In [4]:
set_all_seeds(args.seed)
device = device_from_args(args)

In [6]:
# Data
train_loader,valid_loader, test_loader, N_max, M_max = build_dataloaders(
    args,None,None, for_pretraining=True
)

args Namespace(encoder_path='/home/joachim-verschelde/Repos/KKT_MPNN/src/experiments/kkt_gnn_node_pretraining_experiments/run_gnn_policy-gv=2-lv=8-slice=512-point=13-lmbd=0.256-std=0.0-dim=128-l_mask=0.6-g_mask=0.05-l_e_mask=0.05-g_e_mask=0.6-dp=0.05-l_dim=128-conv=gatv2-heads=4_20260115_175620/best_encoder.pt', devices='0', batch_size=80, num_workers=0, seed=42, max_graphs_per_class=7500, pooling='mean_all', method='pca_tsne', pca_dim=50, perplexity=30.0, finetune_mode='full', use_bipartite_graphs=True, problems=['CA', 'SC', 'IS', 'CFL'], is_sizes=[200], ca_sizes=[100], sc_sizes=[200], cfl_sizes=[200], rnd_sizes=[200], n_instances=50000, data_root='../data/instances', val_split=0.15, lejepa_n_global_views=2, lejepa_n_local_views=8, sigreg_slices=512, sigreg_points=13, lejepa_lambda=0.256, lejepa_std_loss_weight=0.0, embedding_size=128, cons_nfeats=4, edge_nfeats=1, var_nfeats=18, num_emb_type='periodic', num_emb_bins=32, num_emb_freqs=16, lejepa_local_mask=0.6, lejepa_global_mask=0.05

In [7]:
model = GNNPolicy(args).to(device)
model.load_model_and_encoder(args, logger)
model = model.to(device)
model.eval()

2026-01-27 20:42:36.264 | INFO     | models.base:load_model_and_encoder:63 - Encoder unfrozen. Training encoder + heads.
2026-01-27 20:42:36.265 | INFO     | models.base:load_model_and_encoder:69 - Trainable parameters: total=588,552 | encoder=555,272 | heads=33,280


GNNPolicy(
  (sigreg): SigRegWrapper(
    (loss): SlicingUnivariateTest(
      (univariate_test): EppsPulley()
    )
  )
  (encoder): GNNEncoder(
    (cons_num_emb): Sequential(
      (0): PeriodicEmbeddings(
        (periodic): _Periodic()
        (linear): Linear(in_features=32, out_features=24, bias=True)
        (activation): ReLU()
      )
      (1): Flatten(start_dim=1, end_dim=-1)
    )
    (var_num_emb): Sequential(
      (0): PeriodicEmbeddings(
        (periodic): _Periodic()
        (linear): Linear(in_features=32, out_features=24, bias=True)
        (activation): ReLU()
      )
      (1): Flatten(start_dim=1, end_dim=-1)
    )
    (edge_num_emb): Sequential(
      (0): PeriodicEmbeddings(
        (periodic): _Periodic()
        (linear): Linear(in_features=32, out_features=24, bias=True)
        (activation): ReLU()
      )
      (1): Flatten(start_dim=1, end_dim=-1)
    )
    (cons_proj): Sequential(
      (0): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
      (1)

In [8]:

def pooled_graph_embeddings(
    z: torch.Tensor,
    batch: Batch,
    pooling: str = "concat",
) -> torch.Tensor:
    """
    z: [sum_m + sum_n, D] where constraints are first, vars second
    pooling:
      - "mean_all": mean over all nodes (cons+var), producing [B, D]
      - "concat": concat(mean(cons), mean(var)), producing [B, 2D]
    """
    cb = batch.constraint_features_batch  # [sum_m]
    vb = batch.variable_features_batch  # [sum_n]
    B = int(batch.num_graphs)

    m_tot = int(cb.numel())
    ce = z[:m_tot]  # [sum_m, D]
    ve = z[m_tot:]  # [sum_n, D]

    def mean_by_graph(x: torch.Tensor, idx: torch.Tensor) -> torch.Tensor:
        # x: [N, D], idx: [N] in [0..B-1]
        out = x.new_zeros((B, x.size(-1)))
        out.index_add_(0, idx, x)
        counts = x.new_zeros((B,), dtype=torch.float32)
        ones = torch.ones((idx.size(0),), device=x.device, dtype=torch.float32)
        counts.index_add_(0, idx, ones)
        out = out / counts.clamp_min(1.0).unsqueeze(-1)
        return out

    cons_mean = mean_by_graph(ce, cb)  # [B, D]
    var_mean = mean_by_graph(ve, vb)  # [B, D]

    if pooling == "concat":
        return torch.cat([cons_mean, var_mean], dim=-1)  # [B, 2D]
    elif pooling == "mean_all":
        # weighted mean by node count
        out = cons_mean + var_mean
        # NOTE: above isn't weighted. If you want weighted by counts:
        # (sum_cons + sum_var)/(count_cons+count_var) -> implement sums & counts.
        # For visualization, concat is usually better; keep mean_all simple.
        return out * 0.5
    else:
        raise ValueError(f"Unknown pooling: {pooling}")
    

PROBLEM_CLASSES = ["CA", "IS", "CFL", "SC"]

def infer_problem_class(sample_path: str) -> str:
    parts = Path(sample_path).parts
    for p in PROBLEM_CLASSES:
        if p in parts:
            return p
    # fallback: try substring match (rarely needed)
    sp = str(sample_path)
    for p in PROBLEM_CLASSES:
        if f"/{p}/" in sp or f"\\{p}\\" in sp:
            return p
    return "UNK"

def pca_numpy(X: np.ndarray, n_components: int) -> np.ndarray:
    Xc = X - X.mean(axis=0, keepdims=True)
    # SVD-based PCA
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    W = Vt[:n_components].T  # [D, n_components]
    return Xc @ W


def plot_2d(
    Y: np.ndarray,
    labels: List[str],
    out_path: str,
    title: str,
) -> None:
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111)

    classes = [c for c in PROBLEM_CLASSES if c in set(labels)]
    colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    color_map = {c: colors[i % len(colors)] for i, c in enumerate(classes)}

    for c in classes:
        idx = [i for i, lab in enumerate(labels) if lab == c]
        if not idx:
            continue
        pts = Y[idx]
        ax.scatter(pts[:, 0], pts[:, 1], s=10, alpha=0.8, label=c, c=color_map[c])

    ax.set_title(title)
    ax.set_xlabel("dim-1")
    ax.set_ylabel("dim-2")
    ax.legend(loc="best", markerscale=2, frameon=True)
    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)

    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)


def plot_3d(
    Y: np.ndarray,
    labels: List[str],
    out_path: str,
    title: str,
) -> None:
    fig = plt.figure(figsize=(9, 7))
    ax = fig.add_subplot(111, projection="3d")

    classes = [c for c in PROBLEM_CLASSES if c in set(labels)]
    colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    color_map = {c: colors[i % len(colors)] for i, c in enumerate(classes)}

    for c in classes:
        idx = [i for i, lab in enumerate(labels) if lab == c]
        if not idx:
            continue
        pts = Y[idx]
        ax.scatter(
            pts[:, 0], pts[:, 1], pts[:, 2], s=10, alpha=0.8, label=c, c=color_map[c]
        )

    ax.set_title(title)
    ax.set_xlabel("dim-1")
    ax.set_ylabel("dim-2")
    ax.set_zlabel("dim-3")
    ax.legend(loc="best", markerscale=2, frameon=True)

    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)

In [9]:
# ----- Extract embeddings -----
by_class: Dict[str, List[np.ndarray]] = defaultdict(list)

with torch.no_grad():
    for batch in tqdm(test_loader):
        base = batch["base"].to(device, non_blocking=True)
        embeddings = model.embed([base])[0]

        g_emb = pooled_graph_embeddings(embeddings, base)  # [B, D'] (D' = 2D if concat)

        # pull sample paths to label each graph
        sample_paths = getattr(base, "sample_path", None)
        if sample_paths is None:
            # fallback: GraphDataset uses sample_path attribute, so this should exist
            raise RuntimeError("Batch has no sample_path attribute; cannot infer problem class.")

        # sample_paths is a list of strings length B
        for i, sp in enumerate(sample_paths):
            cls = infer_problem_class(sp)
            if cls in PROBLEM_CLASSES:
                by_class[cls].append(g_emb[i].detach().cpu().numpy())




  0%|          | 0/375 [00:00<?, ?it/s]

In [10]:
for key, values in by_class.items():
    print(f"Class {key}: {len(values)} samples., shape: {np.array(values).shape}")

Class CA: 7500 samples., shape: (7500, 256)
Class CFL: 7500 samples., shape: (7500, 256)
Class IS: 7500 samples., shape: (7500, 256)
Class SC: 7500 samples., shape: (7500, 256)


In [11]:

# Downsample per class to keep t-SNE tractable and balance classes
final_embs: List[np.ndarray] = []
final_labels: List[str] = []
for cls in PROBLEM_CLASSES:
    vecs = by_class.get(cls, [])
    if not vecs:
        print(f"[WARN] No samples collected for class {cls}")
        continue
    if len(vecs) > args.max_graphs_per_class:
        idx = np.random.permutation(len(vecs))[: args.max_graphs_per_class]
        vecs = [vecs[i] for i in idx]
    final_embs.extend(vecs)
    final_labels.extend([cls] * len(vecs))
    print(f"[INFO] Using {len(vecs)} graphs for class {cls}")

X = np.stack(final_embs, axis=0)  # [N, D']
print(f"[INFO] Total graphs used: {X.shape[0]}, emb dim: {X.shape[1]}")

# Standardize (helps both PCA and t-SNE)
X = (X - X.mean(axis=0, keepdims=True)) / (X.std(axis=0, keepdims=True) + 1e-6)


# ----- PCA (2D + 3D) -----
if args.method in ("pca", "pca_tsne"):
    Y2 = pca_numpy(X, n_components=2)
    plot_2d(
        Y2,
        final_labels,
        out_path=str(out_dir / "pca_2d.png"),
        title=f"PCA 2D (pool={args.pooling})",
    )
    print(f"[OK] Saved {out_dir / 'pca_2d.png'}")

    Y3 = pca_numpy(X, n_components=3)
    plot_3d(
        Y3,
        final_labels,
        out_path=str(out_dir / "pca_3d.png"),
        title=f"PCA 3D (pool={args.pooling})",
    )
    print(f"[OK] Saved {out_dir / 'pca_3d.png'}")

# ----- t-SNE (2D + 3D) -----
if args.method in ("tsne", "pca_tsne"):
    from sklearn.decomposition import PCA
    from sklearn.manifold import TSNE

    # PCA pre-reduction for speed/stability
    pca_dim = int(min(args.pca_dim, X.shape[1], max(2, X.shape[0] - 1)))
    Xp = PCA(n_components=pca_dim, random_state=args.seed).fit_transform(X)

    # perplexity constraint: must be < N
    N = Xp.shape[0]
    perplex = float(args.perplexity)
    perplex = max(5.0, min(perplex, (N - 1) / 3.0))
    print(f"[INFO] t-SNE using perplexity={perplex:.1f}, PCA pre-dim={pca_dim}")

    tsne2 = TSNE(
        n_components=2,
        perplexity=perplex,
        init="pca",
        learning_rate="auto",
        random_state=args.seed,
        n_iter=1500,
    )
    Y2 = tsne2.fit_transform(Xp)
    plot_2d(
        Y2,
        final_labels,
        out_path=str(out_dir / "tsne_2d.png"),
        title=f"t-SNE 2D (PCA{pca_dim}, pool={args.pooling})",
    )
    print(f"[OK] Saved {out_dir / 'tsne_2d.png'}")

    tsne3 = TSNE(
        n_components=3,
        perplexity=perplex,
        init="pca",
        learning_rate="auto",
        random_state=args.seed,
        n_iter=1500,
    )
    Y3 = tsne3.fit_transform(Xp)
    plot_3d(
        Y3,
        final_labels,
        out_path=str(out_dir / "tsne_3d.png"),
        title=f"t-SNE 3D (PCA{pca_dim}, pool={args.pooling})",
    )
    print(f"[OK] Saved {out_dir / 'tsne_3d.png'}")

# Save reduced-data as CSV-friendly numpy for later use
np.save(out_dir / "graph_embeddings.npy", X)
with open(out_dir / "labels.json", "w") as f:
    json.dump(final_labels, f)
print(f"[OK] Saved embeddings to {out_dir / 'graph_embeddings.npy'} and labels.json")


[INFO] Using 7500 graphs for class CA
[INFO] Using 7500 graphs for class IS
[INFO] Using 7500 graphs for class CFL
[INFO] Using 7500 graphs for class SC
[INFO] Total graphs used: 30000, emb dim: 256
[OK] Saved results/pretrained_embeddings/pca_2d.png
[OK] Saved results/pretrained_embeddings/pca_3d.png
[INFO] t-SNE using perplexity=30.0, PCA pre-dim=50


/home/joachim-verschelde/miniconda3/envs/graph-aug/lib/python3.9/site-packages/sklearn/manifold/_t_sne.py:1164: FutureWarning: 'n_iter' was renamed to 'max_iter' in version 1.5 and will be removed in 1.7.
  warnings.warn(


[OK] Saved results/pretrained_embeddings/tsne_2d.png


/home/joachim-verschelde/miniconda3/envs/graph-aug/lib/python3.9/site-packages/sklearn/manifold/_t_sne.py:1164: FutureWarning: 'n_iter' was renamed to 'max_iter' in version 1.5 and will be removed in 1.7.
  warnings.warn(


[OK] Saved results/pretrained_embeddings/tsne_3d.png
[OK] Saved embeddings to results/pretrained_embeddings/graph_embeddings.npy and labels.json
